# 02 - U-Net Architecture

**Goal:** Understand U-Net - the foundational architecture for medical image segmentation.

---

## Why U-Net?

Published in 2015, U-Net became the standard for medical imaging because:
1. Works well with **small datasets** (common in medical imaging)
2. **Skip connections** preserve spatial detail
3. Simple and effective

Your production SwinUNETR is U-Net + Transformers. Understanding U-Net is the foundation.

## The U-Shape

```
Input Image                                              Output Mask
    │                                                         ▲
    ▼                                                         │
┌─────────┐                                             ┌─────────┐
│ Conv 64 │────────────────────────────────────────────►│ Conv 64 │
└────┬────┘          Skip Connection                    └────▲────┘
     │ Pool                                                  │ UpConv
     ▼                                                       │
┌─────────┐                                             ┌─────────┐
│ Conv 128│────────────────────────────────────────────►│ Conv 128│
└────┬────┘          Skip Connection                    └────▲────┘
     │ Pool                                                  │ UpConv
     ▼                                                       │
┌─────────┐                                             ┌─────────┐
│ Conv 256│────────────────────────────────────────────►│ Conv 256│
└────┬────┘          Skip Connection                    └────▲────┘
     │ Pool                                                  │ UpConv
     ▼                                                       │
┌─────────┐                                             ┌─────────┐
│ Conv 512│────────────────────────────────────────────►│ Conv 512│
└────┬────┘          Skip Connection                    └────▲────┘
     │ Pool                                                  │ UpConv
     ▼                                                       │
   ┌───────────────────────────────────────────────────────┐
   │                    Bottleneck (1024)                   │
   └───────────────────────────────────────────────────────┘
```

**Encoder (left):** Extract features, reduce resolution  
**Decoder (right):** Reconstruct resolution, use skip connections  
**Skip connections:** Copy features from encoder to decoder (preserve details)

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

## Building Blocks

### 1. Double Convolution Block
Each level has two 3x3 convolutions with BatchNorm and ReLU:

In [ ]:
class DoubleConv(nn.Module):
    """Two convolutions with BatchNorm and ReLU."""
    
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.double_conv(x)

# Test
block = DoubleConv(1, 64)
x = torch.randn(1, 1, 64, 64)
print(f"Input: {x.shape}")
print(f"Output: {block(x).shape}")
print(f"Parameters: {sum(p.numel() for p in block.parameters()):,}")

### 2. Encoder Block (Down)
MaxPool to reduce size, then DoubleConv:

In [ ]:
class Down(nn.Module):
    """Downscaling: MaxPool + DoubleConv."""
    
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),  # Halve spatial dimensions
            DoubleConv(in_channels, out_channels)
        )
    
    def forward(self, x):
        return self.maxpool_conv(x)

# Test
down = Down(64, 128)
x = torch.randn(1, 64, 64, 64)
print(f"Input: {x.shape}")
print(f"Output: {down(x).shape}")  # 64->32, channels 64->128

### 3. Decoder Block (Up)
Upsample, concatenate skip connection, then DoubleConv:

In [ ]:
class Up(nn.Module):
    """Upscaling: Upsample + Concatenate skip + DoubleConv."""
    
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # Upsample: double spatial dimensions
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        # After concat with skip: in_channels // 2 + in_channels // 2 = in_channels
        self.conv = DoubleConv(in_channels, out_channels)
    
    def forward(self, x, skip):
        # x: from decoder (lower resolution)
        # skip: from encoder (higher resolution)
        
        x = self.up(x)  # Upsample
        
        # Concatenate along channel dimension
        x = torch.cat([skip, x], dim=1)
        
        return self.conv(x)

# Test
up = Up(256, 128)
x_low = torch.randn(1, 256, 16, 16)   # From decoder
skip = torch.randn(1, 128, 32, 32)    # From encoder

print(f"Input from decoder: {x_low.shape}")
print(f"Skip from encoder: {skip.shape}")
print(f"Output: {up(x_low, skip).shape}")

## Complete U-Net

In [ ]:
class UNet(nn.Module):
    """U-Net for image segmentation."""
    
    def __init__(self, in_channels=1, out_channels=2, features=[64, 128, 256, 512]):
        super().__init__()
        
        # Encoder
        self.encoder = nn.ModuleList()
        self.pool = nn.MaxPool2d(2)
        
        # First encoder block (no pooling before)
        self.first_conv = DoubleConv(in_channels, features[0])
        
        # Rest of encoder
        for i in range(len(features) - 1):
            self.encoder.append(Down(features[i], features[i + 1]))
        
        # Bottleneck
        self.bottleneck = Down(features[-1], features[-1] * 2)
        
        # Decoder
        self.decoder = nn.ModuleList()
        reversed_features = features[::-1]  # [512, 256, 128, 64]
        
        # First decoder (from bottleneck)
        self.decoder.append(Up(features[-1] * 2, features[-1]))
        
        # Rest of decoder
        for i in range(len(reversed_features) - 1):
            self.decoder.append(Up(reversed_features[i], reversed_features[i + 1]))
        
        # Final 1x1 conv to get class predictions
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)
    
    def forward(self, x):
        # Store skip connections
        skip_connections = []
        
        # Encoder path
        x = self.first_conv(x)
        skip_connections.append(x)
        
        for down in self.encoder:
            x = down(x)
            skip_connections.append(x)
        
        # Bottleneck
        x = self.bottleneck(x)
        
        # Decoder path (use skip connections in reverse order)
        skip_connections = skip_connections[::-1]
        
        for i, up in enumerate(self.decoder):
            x = up(x, skip_connections[i])
        
        # Final classification
        return self.final_conv(x)

# Create model
model = UNet(in_channels=1, out_channels=4)  # 4 classes
print(model)

In [ ]:
# Test forward pass
x = torch.randn(2, 1, 128, 128)  # Batch of 2, grayscale, 128x128
output = model(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## Tracing Through the Network

Let's see how shapes change at each step:

In [ ]:
# Trace shapes through the network
x = torch.randn(1, 1, 128, 128)
print(f"Input: {x.shape}\n")

print("=== ENCODER ===")
# First conv
x = model.first_conv(x)
skip1 = x
print(f"After first_conv: {x.shape}")

# Encoder blocks
x = model.encoder[0](x)
skip2 = x
print(f"After down1: {x.shape}")

x = model.encoder[1](x)
skip3 = x
print(f"After down2: {x.shape}")

x = model.encoder[2](x)
skip4 = x
print(f"After down3: {x.shape}")

# Bottleneck
x = model.bottleneck(x)
print(f"\nBottleneck: {x.shape}\n")

print("=== DECODER ===")
x = model.decoder[0](x, skip4)
print(f"After up1 + skip: {x.shape}")

x = model.decoder[1](x, skip3)
print(f"After up2 + skip: {x.shape}")

x = model.decoder[2](x, skip2)
print(f"After up3 + skip: {x.shape}")

x = model.decoder[3](x, skip1)
print(f"After up4 + skip: {x.shape}")

# Final
x = model.final_conv(x)
print(f"\nOutput: {x.shape}")

## Why Skip Connections Matter

Without skip connections, the decoder only has compressed features to work with. Fine details (edges, boundaries) are lost.

Skip connections provide:
- **High-resolution features** from encoder
- **Localization information** (where things are)
- Better gradients during training (similar to ResNet)

In [ ]:
# Visualize: what the network "sees" at different levels

# Create a test image with fine details
import numpy as np

size = 128
image = np.zeros((size, size), dtype=np.float32)

# Add a circle
y, x = np.ogrid[:size, :size]
circle = np.sqrt((x - 64)**2 + (y - 64)**2) <= 30
image[circle] = 0.8

# Add fine edge details
image[60:68, 30:98] = 0.5
image[30:98, 60:68] = 0.5

test_input = torch.tensor(image).unsqueeze(0).unsqueeze(0)  # (1, 1, 128, 128)

# Get features at different encoder levels
with torch.no_grad():
    feat1 = model.first_conv(test_input)
    feat2 = model.encoder[0](feat1)
    feat3 = model.encoder[1](feat2)
    feat4 = model.encoder[2](feat3)
    bottleneck = model.bottleneck(feat4)

# Visualize
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

axes[0, 0].imshow(image, cmap='gray')
axes[0, 0].set_title(f'Input\n{test_input.shape}')

axes[0, 1].imshow(feat1[0, 0].numpy(), cmap='gray')
axes[0, 1].set_title(f'Encoder Level 1\n{feat1.shape}')

axes[0, 2].imshow(feat2[0, 0].numpy(), cmap='gray')
axes[0, 2].set_title(f'Encoder Level 2\n{feat2.shape}')

axes[1, 0].imshow(feat3[0, 0].numpy(), cmap='gray')
axes[1, 0].set_title(f'Encoder Level 3\n{feat3.shape}')

axes[1, 1].imshow(feat4[0, 0].numpy(), cmap='gray')
axes[1, 1].set_title(f'Encoder Level 4\n{feat4.shape}')

axes[1, 2].imshow(bottleneck[0, 0].numpy(), cmap='gray')
axes[1, 2].set_title(f'Bottleneck\n{bottleneck.shape}')

for ax in axes.flat:
    ax.axis('off')

plt.suptitle('Feature maps at different encoder levels\n(spatial detail is lost deeper in the network)', fontsize=12)
plt.tight_layout()
plt.show()

## U-Net Variants

| Variant | Change | Use case |
|---------|--------|----------|
| **U-Net** | Original | 2D medical imaging |
| **3D U-Net** | 3D convolutions | CT/MRI volumes (your case) |
| **Attention U-Net** | Attention gates | Focus on relevant regions |
| **U-Net++** | Nested skip connections | Better feature fusion |
| **SwinUNETR** | Swin Transformer encoder | Your production model |

Your production code uses **SwinUNETR**: U-Net decoder with a Swin Transformer encoder (instead of conv encoder).

## 3D U-Net (For Volumes)

Same architecture, but with 3D operations:

In [ ]:
# 3D version of double conv
class DoubleConv3D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.double_conv(x)

# Test with 3D volume (like your CT scans)
block_3d = DoubleConv3D(1, 24)  # 24 features like SwinUNETR
volume = torch.randn(1, 1, 96, 96, 96)  # Your input shape!

print(f"Input volume: {volume.shape}")
print(f"Output: {block_3d(volume).shape}")
print(f"Parameters: {sum(p.numel() for p in block_3d.parameters()):,}")

## Connection to Production Code

Your `SwinUNETR` from MONAI follows the same principle:

```python
from monai.networks.nets import SwinUNETR

model = SwinUNETR(
    img_size=[96, 96, 96],    # Input volume size
    in_channels=1,            # Grayscale CT
    out_channels=nb_classes,  # Number of classes
    feature_size=24,          # Base number of features
)
```

The difference: SwinUNETR uses **Swin Transformer** as the encoder instead of convolutions. We'll cover that in Phase 3.

## Summary

| Component | Purpose |
|-----------|--------|
| **Encoder** | Extract features, reduce resolution |
| **Bottleneck** | Compressed representation |
| **Decoder** | Reconstruct resolution |
| **Skip connections** | Preserve spatial detail |
| **Final conv (1x1)** | Map features to class predictions |

**Key insight:** The U-shape lets the network learn *what* is there (encoder) and *where* it is (decoder + skips).

**Next:** Train a U-Net on real data.